# GTU Mimari Lejant - Colab Training

Bu notebook Roboflow YOLOv11 instance segmentation export'u ile ilk YOLO segmentation baseline modelini egitir.

Colab ayari: `Runtime -> Change runtime type -> GPU`. GPU onceligi: A100 > L4 > T4.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Repo'yu Klonla

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r /content/lejanter_doga_vlm_codex/requirements.txt

## 2. Roboflow Zip'i Yukle veya Drive'dan Kopyala

Asagidaki seceneklerden sadece birini calistir:

1. Bilgisayardan upload.
2. Kendi MyDrive path'inden kopyalama.
3. Paylasilan Drive klasorunden `dataset/` ve `weights/` klasorlerini indirme.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print('Uploaded:', zip_name)

## 2B. Google Drive'dan Zip Kopyala

Upload yerine dataset zip dosyasini Google Drive'dan almak istersen bu hucreyi calistir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -f '/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/GTU_MIMARI_LEJANT.yolov11.zip' '/content/GTU_MIMARI_LEJANT.yolov11.zip'
zip_name = '/content/GTU_MIMARI_LEJANT.yolov11.zip'
print('Dataset zip:', zip_name)

## 2C. Paylasilan Drive Klasorunden Dataset ve Weights Al

Paylasilan Drive klasorunde `dataset/` ve `weights/` alt klasorleri varsa bu hucreyi kullan. `dataset/` icindeki zip otomatik bulunur; `weights/` icindeki `.pt` dosyalari da Colab icine kopyalanir.

In [ ]:
from pathlib import Path
import shutil

SHARED_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
SHARED_DOWNLOAD_DIR = Path('/content/shared_drive_lejant')
if SHARED_DOWNLOAD_DIR.exists():
    shutil.rmtree(SHARED_DOWNLOAD_DIR)
SHARED_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

!pip install -q gdown
!gdown --folder '{SHARED_DRIVE_FOLDER_URL}' -O /content/shared_drive_lejant --remaining-ok

print('Downloaded files:')
for path in sorted(SHARED_DOWNLOAD_DIR.rglob('*')):
    print(path)

dataset_zips = list((SHARED_DOWNLOAD_DIR / 'dataset').rglob('*.zip')) if (SHARED_DOWNLOAD_DIR / 'dataset').exists() else list(SHARED_DOWNLOAD_DIR.rglob('*.zip'))
if dataset_zips:
    zip_name = str(dataset_zips[0])
    print('Dataset zip:', zip_name)
else:
    zip_name = None
    print('No dataset zip found. This is OK if you only want to use Drive weights/test images. Run upload or 2B before dataset extraction if training/evaluation split is needed.')

weights_src = SHARED_DOWNLOAD_DIR / 'weights'
if not weights_src.exists() and any(SHARED_DOWNLOAD_DIR.glob('*.pt')):
    weights_src = SHARED_DOWNLOAD_DIR
weights_dst = Path('/content/lejanter_doga_vlm_codex/drive_weights')
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)
if weights_src.exists():
    for pt in weights_src.rglob('*.pt'):
        rel = pt.relative_to(weights_src)
        nested_dst = weights_dst / rel
        nested_dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pt, nested_dst)

        # If the file is outputs/runs/<model>/weights/best.pt, also create a unique flat alias.
        if pt.name in {'best.pt', 'last.pt'} and pt.parent.name == 'weights' and pt.parent.parent != weights_src:
            alias = weights_dst / f'{pt.parent.parent.name}-{pt.name}'
            shutil.copy2(pt, alias)
    print('Copied weights to:', weights_dst)
    for pt in sorted(weights_dst.rglob('*.pt')):
        print('-', pt)
else:
    print('No weights folder found in shared Drive download')

test_src = SHARED_DOWNLOAD_DIR / 'test'
if not test_src.exists():
    # If the shared folder itself contains images, use it as the test source.
    has_root_images = any(p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'} for p in SHARED_DOWNLOAD_DIR.iterdir())
    if has_root_images:
        test_src = SHARED_DOWNLOAD_DIR
test_dst = Path('/content/lejanter_doga_vlm_codex/data/raw/drive_test')
if test_dst.exists():
    shutil.rmtree(test_dst)
test_dst.mkdir(parents=True, exist_ok=True)
if test_src.exists():
    for image in test_src.rglob('*'):
        if image.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}:
            shutil.copy2(image, test_dst / image.name)
    print('Copied test images to:', test_dst)
    for image in sorted(test_dst.glob('*')):
        print('-', image)
else:
    print('No test folder found in shared Drive download')

reports_drive_dir = SHARED_DOWNLOAD_DIR / 'reports'
print('Downloaded reports folder candidate:', reports_drive_dir)

Not: Upload hucrelerinden sonra `zip_name` degiskeni set edilmis olmali. Dataset acma hucreleri bu degiskeni kullanir.

## 3. Dataset'i Ac ve Split Hazirla

In [ ]:
from pathlib import Path
import shutil
import yaml

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
dataset_root = PROJECT_DIR / 'data/roboflow/gtu-mimari-lejant'
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True, exist_ok=True)

!unzip -q "{zip_name}" -d /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant

def image_count(split):
    image_dir = dataset_root / split / 'images'
    if not image_dir.exists():
        return 0
    return sum(1 for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'})

print('Before split:', {s: image_count(s) for s in ['train', 'valid', 'test']})

# Roboflow export bazen sadece train split'i ile gelir. Valid/test yoksa veya bossa olustur.
if image_count('valid') == 0 or image_count('test') == 0:
    shutil.rmtree(dataset_root / 'valid', ignore_errors=True)
    shutil.rmtree(dataset_root / 'test', ignore_errors=True)
    !cd /content/lejanter_doga_vlm_codex && python scripts/split_yolo_dataset.py --root /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant --valid 0.1 --test 0.1
else:
    print('valid/test split already exists and has images')

print('After split:', {s: image_count(s) for s in ['train', 'valid', 'test']})
assert image_count('train') > 0, 'train/images is empty'
assert image_count('valid') > 0, 'valid/images is empty'
assert image_count('test') > 0, 'test/images is empty'

# Colab icin tum path'ler mutlak.
cfg_path = PROJECT_DIR / 'configs/elements_dataset.yaml'
cfg = yaml.safe_load(cfg_path.read_text())
cfg['path'] = str(dataset_root)
colab_cfg_path = PROJECT_DIR / 'configs/elements_colab.yaml'
colab_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(colab_cfg_path.read_text())

In [ ]:
!find /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant -maxdepth 3 -type f | awk -F/ '{print $(NF-2) "/" $(NF-1)}' | sort | uniq -c
!sed -n '1,80p' /content/lejanter_doga_vlm_codex/configs/elements_colab.yaml
!test -d /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant/train/images && test -d /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant/valid/images && test -d /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant/test/images && echo 'Dataset folders OK'

## 4. Ilk Baseline Egitimi

T4 GPU'da bellek sorunu olursa `yolo11s-seg.pt` yerine `yolo11n-seg.pt`, `imgsz=768` kullan.

In [ ]:
!cd /content/lejanter_doga_vlm_codex && python scripts/train_yolo.py \
  --data /content/lejanter_doga_vlm_codex/configs/elements_colab.yaml \
  --model yolo11s-seg.pt \
  --task segment \
  --name elements-seg-v1 \
  --epochs 100 \
  --imgsz 1024 \
  --batch -1 \
  --project /content/lejanter_doga_vlm_codex/outputs/runs \
  --device 0

## 4B. Kontrollu Augmentation Deneyi

Bu hucre opsiyonel ikinci deneydir. Ilk baseline bittikten sonra calistir. Amac default augmentation yerine mimari cephe icin daha kontrollu augmentation denemektir.

In [ ]:
!cd /content/lejanter_doga_vlm_codex && yolo segment train \
  model=yolo11s-seg.pt \
  data=/content/lejanter_doga_vlm_codex/configs/elements_colab.yaml \
  project=/content/lejanter_doga_vlm_codex/outputs/runs \
  name=elements-seg-v2-aug-controlled \
  epochs=300 \
  patience=150 \
  imgsz=1024 \
  batch=-1 \
  device=0 \
  mosaic=0.3 \
  erasing=0.0 \
  scale=0.3 \
  translate=0.05 \
  hsv_h=0.01 \
  hsv_s=0.4 \
  hsv_v=0.3 \
  fliplr=0.5 \
  flipud=0.0 \
  perspective=0.0

## 4C. Drive'dan Kayitli Modeli Kullan

Egitim yapmadan, Drive'daki `weights/` klasorunden gelen `.pt` dosyasiyla inference yapmak istersen bu hucreyi calistir. Varsayilan olarak v3 modelini arar.

In [ ]:
from pathlib import Path
import shutil

weights_dir = Path('/content/lejanter_doga_vlm_codex/drive_weights')
candidate_names = [
    'elements-seg-v3-no-erasing-best.pt',
    'elements-seg-v3-no-erasing.pt',
    'best.pt',
]
candidates = []
for name in candidate_names:
    candidates.extend(weights_dir.rglob(name) if weights_dir.exists() else [])
if not candidates and weights_dir.exists():
    candidates = list(weights_dir.rglob('*.pt'))
assert candidates, 'No .pt weight found. Run the shared Drive weights cell first or train a model.'
src = candidates[0]
dst = Path('/content/lejanter_doga_vlm_codex/outputs/runs/elements-seg-v3-no-erasing/weights/best.pt')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
print('Using weight:', src)
print('Copied to:', dst)

## 4D. Tum Modelleri Drive Test Gorsellerinde Karsilastir

Drive `test/` klasorundeki gorselleri v1/v2/v3 weight dosyalariyla test eder. Her model icin ayri klasor ve model adini iceren dosya adlari uretir.

In [ ]:
from pathlib import Path
import shutil
import json

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
test_source = PROJECT_DIR / 'data/raw/drive_test'
assert test_source.exists() and any(test_source.iterdir()), 'No test images found. Run shared Drive cell and add images to Drive test/ folder.'

weights_dir = PROJECT_DIR / 'drive_weights'
model_specs = {
    'v1-default': {
        'run_dir': 'elements-seg-v1',
        'names': ['elements-seg-v1-best.pt', 'elements-seg-v1.pt'],
    },
    'v2-aug-controlled': {
        'run_dir': 'elements-seg-v2-aug-controlled',
        'names': ['elements-seg-v2-aug-controlled-best.pt', 'elements-seg-v2-aug-controlled.pt'],
    },
    'v3-no-erasing': {
        'run_dir': 'elements-seg-v3-no-erasing',
        'names': ['elements-seg-v3-no-erasing-best.pt', 'elements-seg-v3-no-erasing.pt'],
    },
}

def find_weight(spec):
    if not weights_dir.exists():
        return None
    for name in spec['names']:
        hits = list(weights_dir.rglob(name))
        if hits:
            return hits[0]
    run_dir = spec['run_dir']
    patterns = [
        f'{run_dir}/weights/best.pt',
        f'**/{run_dir}/weights/best.pt',
        f'{run_dir}-best.pt',
        f'{run_dir}*best.pt',
    ]
    for pattern in patterns:
        hits = list(weights_dir.glob(pattern))
        if hits:
            return hits[0]
    return None

available = {model: find_weight(spec) for model, spec in model_specs.items()}
available = {model: path for model, path in available.items() if path is not None}
assert available, 'No expected model weights found under drive_weights/'
print('Available models:')
for model, path in available.items():
    print(model, '->', path)

benchmark_root = PROJECT_DIR / 'outputs/drive_test_benchmark'
if benchmark_root.exists():
    shutil.rmtree(benchmark_root)
benchmark_root.mkdir(parents=True, exist_ok=True)

for model_name, weight_path in available.items():
    run_name = f'drive-test-{model_name}'
    out_json = PROJECT_DIR / 'outputs/reports' / f'drive_test_{model_name}.json'
    !cd /content/lejanter_doga_vlm_codex && python scripts/infer_yolo.py \
      --weights {str(weight_path)} \
      --source {str(test_source)} \
      --task segment \
      --out {str(out_json)} \
      --name {run_name} \
      --project /content/lejanter_doga_vlm_codex/outputs/runs \
      --save-visuals \
      --device 0
    src_dir = PROJECT_DIR / 'outputs/runs' / run_name
    dst_dir = benchmark_root / model_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    for image in src_dir.glob('*'):
        if image.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            shutil.copy2(image, dst_dir / f'{model_name}__{image.name}')
    shutil.copy2(out_json, benchmark_root / f'{model_name}__detections.json')

summary = {model: str(path) for model, path in available.items()}
(benchmark_root / 'model_weights_used.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Benchmark outputs:', benchmark_root)
!find /content/lejanter_doga_vlm_codex/outputs/drive_test_benchmark -maxdepth 2 -type f | sort | sed -n '1,200p'

## 5. Test Set Inference

In [ ]:
!rm -rf /content/lejanter_doga_vlm_codex/outputs/runs/infer-elements-test
!cd /content/lejanter_doga_vlm_codex && python scripts/infer_yolo.py \
  --weights /content/lejanter_doga_vlm_codex/outputs/runs/elements-seg-v1/weights/best.pt \
  --source /content/lejanter_doga_vlm_codex/data/roboflow/gtu-mimari-lejant/test/images \
  --task segment \
  --out /content/lejanter_doga_vlm_codex/outputs/reports/elements_test.json \
  --name infer-elements-test \
  --project /content/lejanter_doga_vlm_codex/outputs/runs \
  --save-visuals \
  --device 0

!ls -R /content/lejanter_doga_vlm_codex/outputs/runs/infer-elements-test | sed -n '1,160p'

## 6. Sonuclari Goster

In [ ]:
from IPython.display import Image, display
from pathlib import Path

pred_dir = Path('/content/lejanter_doga_vlm_codex/outputs/runs/infer-elements-test')
if not pred_dir.exists():
    raise FileNotFoundError(f'Prediction folder not found: {pred_dir}')
for image_path in list(pred_dir.glob('*'))[:5]:
    if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        display(Image(filename=str(image_path)))

## 7. Kendi Gorselini Dene

Egitim bittikten sonra yeni bir cephe fotografi yukleyip `best.pt` ile sonucu gorebilirsin. Bir veya birden fazla `.jpg/.png` dosyasi secilebilir.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

custom_dir = Path('/content/lejanter_doga_vlm_codex/data/raw/custom_uploads')
if custom_dir.exists():
    shutil.rmtree(custom_dir)
custom_dir.mkdir(parents=True, exist_ok=True)

custom_uploaded = files.upload()
for name, content in custom_uploaded.items():
    (custom_dir / name).write_bytes(content)

print('Uploaded custom images:')
for path in custom_dir.iterdir():
    print('-', path)

In [ ]:
!rm -rf /content/lejanter_doga_vlm_codex/outputs/runs/infer-custom-upload
!cd /content/lejanter_doga_vlm_codex && python scripts/infer_yolo.py \
  --weights /content/lejanter_doga_vlm_codex/outputs/runs/elements-seg-v1/weights/best.pt \
  --source /content/lejanter_doga_vlm_codex/data/raw/custom_uploads \
  --task segment \
  --out /content/lejanter_doga_vlm_codex/outputs/reports/custom_uploads.json \
  --name infer-custom-upload \
  --project /content/lejanter_doga_vlm_codex/outputs/runs \
  --save-visuals \
  --device 0

!ls -R /content/lejanter_doga_vlm_codex/outputs/runs/infer-custom-upload | sed -n '1,160p'

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import json

pred_dir = Path('/content/lejanter_doga_vlm_codex/outputs/runs/infer-custom-upload')
if not pred_dir.exists():
    raise FileNotFoundError(f'Prediction folder not found: {pred_dir}')
for image_path in sorted(pred_dir.glob('*')):
    if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        display(Image(filename=str(image_path)))

custom_json = Path('/content/lejanter_doga_vlm_codex/outputs/reports/custom_uploads.json')
if custom_json.exists():
    data = json.loads(custom_json.read_text())
    print(json.dumps(data[:1], ensure_ascii=False, indent=2)[:3000])

In [ ]:
!cd /content/lejanter_doga_vlm_codex && zip -r /content/custom_upload_predictions.zip outputs/runs/infer-custom-upload outputs/reports/custom_uploads.json
files.download('/content/custom_upload_predictions.zip')

## 8. Model ve Raporlari Indir

In [ ]:
!cd /content/lejanter_doga_vlm_codex && zip -r /content/gtu_elements_results.zip outputs/runs/elements-seg-v1 outputs/runs/infer-elements-test outputs/reports/elements_test.json
!if [ -d /content/lejanter_doga_vlm_codex/outputs/runs/elements-seg-v2-aug-controlled ]; then cd /content/lejanter_doga_vlm_codex && zip -r /content/gtu_elements_results.zip outputs/runs/elements-seg-v2-aug-controlled; fi
files.download('/content/gtu_elements_results.zip')

## 9. Sonuclari Drive Reports Klasorune Kopyala

Egitim raporlari, metrikler, config dosyalari ve test gorsellerini Drive `reports/` klasorune kopyalar.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import datetime

drive.mount('/content/drive')

# If the shared Drive folder is mounted somewhere else, change only this path.
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = REPORTS_DIR / f'yolo_reports_{stamp}'
export_dir.mkdir(parents=True, exist_ok=True)

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
items = [
    PROJECT_DIR / 'outputs/runs/elements-seg-v1',
    PROJECT_DIR / 'outputs/runs/elements-seg-v2-aug-controlled',
    PROJECT_DIR / 'outputs/runs/elements-seg-v3-no-erasing',
    PROJECT_DIR / 'outputs/drive_test_benchmark',
    PROJECT_DIR / 'outputs/reports',
    PROJECT_DIR / 'configs/elements_colab.yaml',
]
for item in items:
    if item.exists():
        dst = export_dir / item.name
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

zip_base = REPORTS_DIR / f'yolo_reports_{stamp}'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=REPORTS_DIR, base_dir=f'yolo_reports_{stamp}')
print('Reports copied to:', export_dir)
print('Zip:', zip_path)